# 00 · 환경 검증

**목적** 학습을 시작해도 되는 상태인지 한 번에 판정한다.

확인 항목

1. 패키지 버전 (torch / timm / ultralytics / albumentations …)
2. CUDA 를 **실제로** 잡는지 — `is_available()` 만이 아니라 matmul + AMP 까지 실행
3. 승계 대상 경로 존재 여부 (D-06 / D-12)
4. 원본 데이터셋 존재 여부
5. 한글 CSV round-trip (PLAN P8 — Windows cp949)
6. configs 로드

**통과 조건** `status == "ok"` 이고 `blockers` 가 비어 있을 것.

로직은 전부 `src/banggoot/envcheck.py` 에 있다. 이 노트북은 호출만 한다.

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "src"))

print("repo root:", REPO_ROOT)

repo root: C:\Users\SSAFY\Documents\ssafy_jc\sub_pjt\banggoot_model


## 1. 전체 검사 실행

In [2]:
from banggoot import envcheck

report = envcheck.run()
print(envcheck.summary(report))

status: ok

패키지 20/20 ok (requirements-train.txt 기준)
torch 2.6.0+cu124 / cuda_build 12.4 / NVIDIA GeForce RTX 4050 Laptop GPU 6141MB -> ok
승계 경로 18/18 존재
인코딩: preferred=cp949 dacon=19/19 roundtrip=True

차단 이슈 없음. 노트북 01 진행 가능.

## 2. 패키지 상세

`version_mismatch` 는 즉시 차단은 아니지만, 기존 저장소에서 실측 검증된 조합과
다르다는 뜻이므로 결과 재현성에 영향을 줄 수 있다.

In [3]:
import pandas as pd

pd.DataFrame(report["packages"])

,package,op,want,got,status,import_ok
0,torch,==,2.6.0+cu124,2.6.0+cu124,ok,True
1,torchvision,==,0.21.0+cu124,0.21.0+cu124,ok,True
2,timm,==,1.0.22,1.0.22,ok,True
3,ultralytics,==,8.4.112,8.4.112,ok,True
4,numpy,==,2.4.6,2.4.6,ok,True
5,pandas,==,3.0.5,3.0.5,ok,True
6,pillow,==,12.3.0,12.3.0,ok,True
7,opencv-python,==,5.0.0.93,5.0.0.93,ok,True
8,scikit-learn,==,1.9.0,1.9.0,ok,True
9,PyYAML,==,6.0.3,6.0.3,ok,True


## 3. GPU

기준 실측 (기존 저장소): RTX 4050 Laptop 6GB 에서 EfficientNet-B0 224px 학습
약 7분 / peak VRAM 1.43GB. 320px batch 32 도 여유가 있다.

In [4]:
report["torch"]

{'torch': '2.6.0+cu124',
 'cuda_build': '12.4',
 'cuda_available': True,
 'device_name': 'NVIDIA GeForce RTX 4050 Laptop GPU',
 'capability': 'sm_89',
 'total_vram_mb': 6141,
 'matmul_ok': True,
 'peak_vram_mb': 48.12,
 'status': 'ok'}

## 4. 승계 경로

`kind=inherit` 는 전부 존재해야 한다. 특히 `origin_decisions` 가 없으면
**강등 이전 origin 을 상속**하게 되므로 반드시 확인한다 (D-12).

In [5]:
df = pd.DataFrame(report["legacy_paths"])
display(df[df["kind"] == "inherit"][["key", "exists", "path"]])
display(df[df["kind"] != "inherit"][["kind", "key", "exists", "required"]])

,key,exists,path
3,dataset_inventory,True,C:\Users\SSAFY\Documents\ssafy_jc\sub_pjt\defe...
4,inventory_summary,True,C:\Users\SSAFY\Documents\ssafy_jc\sub_pjt\defe...
5,split_manifest,True,C:\Users\SSAFY\Documents\ssafy_jc\sub_pjt\defe...
6,duplicate_groups,True,C:\Users\SSAFY\Documents\ssafy_jc\sub_pjt\defe...
7,image_hashes,True,C:\Users\SSAFY\Documents\ssafy_jc\sub_pjt\defe...
8,origin_decisions,True,C:\Users\SSAFY\Documents\ssafy_jc\sub_pjt\defe...
9,origin_overrides,True,C:\Users\SSAFY\Documents\ssafy_jc\sub_pjt\defe...
10,origin_ok_marker,True,C:\Users\SSAFY\Documents\ssafy_jc\sub_pjt\defe...
11,review_manifest,True,C:\Users\SSAFY\Documents\ssafy_jc\sub_pjt\defe...
12,artifact_decisions,True,C:\Users\SSAFY\Documents\ssafy_jc\sub_pjt\defe...


,kind,key,exists,required
0,base,legacy_root,True,True
1,base,data_raw,True,True
2,base,registry,True,True
18,reference,classifier_l3,True,False
19,reference,experiments,True,False
20,reference,yolo_d0_s0,True,False
21,reference,aihub_positive_crops,True,False


In [6]:
pd.DataFrame(report["raw_datasets"])

,dataset,path,exists
0,dacon_wallpaper,C:\Users\SSAFY\Documents\ssafy_jc\sub_pjt\defe...,True
1,aihub_seoul_aged_housing,C:\Users\SSAFY\Documents\ssafy_jc\sub_pjt\defe...,True
2,kaggle_infrastructure_structural_defects,C:\Users\SSAFY\Documents\ssafy_jc\sub_pjt\defe...,True
3,roboflow_wall_defects,C:\Users\SSAFY\Documents\ssafy_jc\sub_pjt\defe...,True
4,roboflow_wallpaper_kr,C:\Users\SSAFY\Documents\ssafy_jc\sub_pjt\defe...,True
5,roboflow_house_defect,C:\Users\SSAFY\Documents\ssafy_jc\sub_pjt\defe...,True


## 5. 인코딩 (P8)

Windows 기본 인코딩은 cp949 다. 한글 라벨 CSV 를 인코딩 지정 없이 읽고 쓰면
라벨이 깨지고, 그 상태로 crop 을 만들면 노트북 02 이후에야 발견된다.

- `dacon_classes` 는 **19**
- `first_header` 는 `source_dataset` (BOM 이 남으면 `\ufeffsource_dataset` 이 된다)

In [7]:
report["encoding"]

{'sys_default_encoding': 'utf-8',
 'preferred_encoding': 'cp949',
 'stdout_encoding': 'UTF-8',
 'io_encoding': 'utf-8-sig',
 'label_mapping_rows': 58,
 'dacon_classes': 19,
 'korean_roundtrip_ok': True,
 'first_header': 'source_dataset',
 'status': 'ok'}

In [8]:
report["configs"]

{'status': 'ok',
 'l1_classes': ['crack',
  'breakage',
  'stain_corrosion',
  'moisture_leak',
  'lifting',
  'mold',
  'finish_damage'],
 'sample_unit': 'crop',
 'normal_is_a_class': False}

## 6. 결과 저장 및 게이트

`artifacts/experiments/env_report.json` 에 저장한다. 이 파일은 git 추적 대상이며
학습 결과의 재현 환경 근거가 된다.

**차단 이슈가 있으면 아래 셀이 실패한다. 노트북 01 로 넘어가지 않는다.**

In [9]:
out = envcheck.save(report)
print("saved:", out)

assert not report["blockers"], "차단 이슈:\n" + "\n".join(report["blockers"])
print("\n환경 검증 통과. 노트북 01 진행 가능.")

saved:

C:\Users\SSAFY\Documents\ssafy_jc\sub_pjt\banggoot_model\artifacts\experiments\env_report.json


환경 검증 통과. 노트북 01 진행 가능.